# OpenVLA × SelfIE — Colab demo

End-to-end: load OpenVLA-7B on a Colab GPU and get natural-language interpretations
of its hidden embeddings using a SelfIE-style hook pipeline.

**Runtime setup:** Runtime → Change runtime type → **A100 GPU** (preferred) or **V100 GPU** (fallback), and **High-RAM** if available.

**Run cells top to bottom.** First run downloads ~15 GB of OpenVLA weights (5-10 minutes). If you mount Drive (cell 2), subsequent sessions load from cache in ~30 seconds.


## Cell 1 — Verify GPU

Confirm you have an A100 or V100 with at least 16 GB free.

In [3]:
!nvidia-smi

Thu Apr 23 18:17:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.73                 Driver Version: 580.92         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Quadro T2000                   On  |   00000000:01:00.0 Off |                  N/A |
| N/A   57C    P8              4W /   60W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 2 — (Optional) Mount Google Drive for persistent model cache

Skip this cell if you don't mind re-downloading the 15 GB OpenVLA checkpoint every session.
If you run it, the download will go to your Drive and load instantly in future sessions.

In [ ]:
# from google.colab import drive
# import os

# drive.mount('/content/drive')
# os.makedirs('/content/drive/MyDrive/hf_cache', exist_ok=True)
# os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
# print('HF cache dir:', os.environ['HF_HOME'])

## Cell 3 — Install pinned dependencies

OpenVLA's HF code enforces specific versions. Installing anything else will trigger a version warning.

In [4]:
!pip install -q "transformers==4.40.1" "tokenizers==0.19.1" "timm==0.9.10" \
               sentencepiece accelerate Pillow datasets


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Cell 4 — (Optional) Install flash-attn

Only works on Ampere or newer (A100, A10, H100, RTX 30xx/40xx). Skip on V100 or T4. Compile takes ~10 minutes. Makes inference ~30% faster but the demo works fine without it.

In [ ]:
# Uncomment if you have an A100 and want flash-attn:
# !pip install -q packaging ninja
# !pip install -q "flash-attn==2.5.5" --no-build-isolation

## Cell 5 — Write the `openvla_selfie` library

This writes the hook-based SelfIE library to the Colab filesystem. It's the same code as in my GitHub repo — embedded here so you don't need to upload a file. Just run the cell.

In [5]:
"""
openvla_selfie.py
=================

Apply SelfIE-style self-interpretation (Chen et al., 2024, https://selfie.cs.columbia.edu/)
to OpenVLA (Kim et al., 2024, https://openvla.github.io/).

OpenVLA = Prismatic VLM (DINOv2 + SigLIP -> MLP projector -> Llama-2 7B).
We want text descriptions of hidden embeddings inside the Llama-2 backbone, where
those embeddings may correspond to image-patch tokens (image token embeddings are
genuinely interesting for robot manipulation) or language-instruction tokens.

SelfIE's idea, boiled down:
  1. Run an ORIGINAL forward pass on the real input and save all layer hidden states
     H[l, t]  (layer l, token position t).
  2. Run an INTERPRETATION forward pass on a fixed prompt like
        "[INST] <PLACEHOLDER> [/INST] Sure, I will summarize the message:"
     where at some early layer k, the hidden states at the placeholder positions are
     OVERWRITTEN with H[l, t]. Let the model generate text. That generated text is
     the interpretation of H[l, t].

The tricky part: SelfIE's repo was written against transformers==4.34.0 and copies
the LLaMA decoder forward wholesale. OpenVLA's HF weights are pinned to
transformers==4.40.1, and modern transformers (4.38+) removed / renamed the private
symbols SelfIE relies on. Concretely, SelfIE's `llama_forward_wrappers.py` breaks
because it uses:
  * `is_flash_attn_available`  -- renamed to `is_flash_attn_2_available` in 4.38+
  * `model.model._prepare_decoder_attention_mask(...)` -- removed in 4.38+ in favor
    of `_prepare_4d_causal_attention_mask(...)` as a free function
  * `padding_mask=` kwarg on `LlamaAttention.forward` -- removed in 4.38+
  * hidden-state tuple layout assumed by `past_key_values[0][0]` -- transformers
    switched to `DynamicCache` objects in 4.36+
See the docstring of `patch_selfie_for_transformers_4_40()` below for the exact
one-line edits needed if you want to use the upstream SelfIE code directly.

The approach taken in this module is cleaner and more robust: we DO NOT use
SelfIE's forward wrappers at all. Instead we implement the same algorithm using
PyTorch forward hooks on the native `LlamaDecoderLayer` modules. This is
transformers-version-agnostic and works on OpenVLA out of the box.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import torch
import torch.nn as nn


# ---------------------------------------------------------------------------
# 1. Interpretation prompt
# ---------------------------------------------------------------------------

@dataclass
class InterpretationPrompt:
    """A fixed prompt with placeholder positions we'll overwrite with a hidden
    embedding during the interpretation forward pass.

    We mirror SelfIE's convention: the prompt is a tuple of (str, 0, 0, ..., 0, str)
    where the integer 0 marks a single placeholder slot. Each placeholder becomes
    a dummy token in the string (here we use the literal token " _") and we record
    the index of that token so we can overwrite its hidden state at layer k.

    Example (the SelfIE default for a Llama-2-chat style interpretation):
        InterpretationPrompt.build(
            tokenizer,
            ("[INST]", 0, 0, 0, 0, 0, "[/INST] Sure, I will summarize the message:"),
        )
    gives 5 placeholder tokens in between [INST] and [/INST].
    """

    input_ids: torch.LongTensor        # shape (1, T_prompt)
    attention_mask: torch.LongTensor   # shape (1, T_prompt)
    placeholder_positions: List[int]   # indices in input_ids[0] that are placeholders
    rendered: str                      # the prompt string (for logging)

    @classmethod
    def build(cls, tokenizer, spec: Sequence, placeholder_str: str = " _") -> "InterpretationPrompt":
        rendered = ""
        placeholder_positions: List[int] = []
        for part in spec:
            if isinstance(part, str):
                rendered += part
            else:
                # record where this placeholder lands in the tokenised sequence
                start = len(tokenizer.encode(rendered, add_special_tokens=False))
                rendered += placeholder_str
                end = len(tokenizer.encode(rendered, add_special_tokens=False))
                placeholder_positions.extend(range(start, end))

        enc = tokenizer(rendered, return_tensors="pt")
        # tokenizer may add BOS at position 0; shift recorded positions accordingly.
        # Simplest robust fix: re-locate placeholders by scanning for the placeholder
        # token id in the final input_ids.
        placeholder_token_ids = set(
            tokenizer.encode(placeholder_str, add_special_tokens=False)
        )
        real_positions = [
            i for i, tid in enumerate(enc.input_ids[0].tolist())
            if tid in placeholder_token_ids
        ]
        # If the naive count matches, prefer that (it preserves SelfIE's semantics
        # when the placeholder happens to appear elsewhere as a normal token).
        if len(real_positions) == len(placeholder_positions):
            placeholder_positions = real_positions
        return cls(
            input_ids=enc.input_ids,
            attention_mask=enc.attention_mask,
            placeholder_positions=placeholder_positions,
            rendered=rendered,
        )


# ---------------------------------------------------------------------------
# 2. Hook-based injector
# ---------------------------------------------------------------------------

def _get_llama_layers(model: nn.Module) -> nn.ModuleList:
    """Locate the list of LlamaDecoderLayer modules inside an OpenVLA or bare
    Llama model. We walk a few known paths so this works for:
      * bare LlamaForCausalLM  (model.model.layers)
      * OpenVLA PrismaticForConditionalGeneration
        (model.language_model.model.layers)
      * raw LlamaModel         (model.layers)
    """
    for path in (
        ("language_model", "model", "layers"),   # OpenVLA
        ("model", "layers"),                     # LlamaForCausalLM
        ("layers",),                             # LlamaModel
    ):
        cur = model
        ok = True
        for attr in path:
            if not hasattr(cur, attr):
                ok = False
                break
            cur = getattr(cur, attr)
        if ok and isinstance(cur, nn.ModuleList):
            return cur
    raise AttributeError(
        "Could not locate LlamaDecoderLayer ModuleList on the given model. "
        "Pass the layer list explicitly if your model has a non-standard structure."
    )


class _Injector:
    """Forward hooks that (a) record every layer's input hidden state during an
    original pass, and (b) overwrite the layer-k hidden state at chosen token
    positions with a provided tensor during an interpretation pass.
    """

    def __init__(self, layers: nn.ModuleList):
        self.layers = layers
        self.handles: List[torch.utils.hooks.RemovableHandle] = []
        self.recorded: Dict[int, torch.Tensor] = {}     # layer_idx -> (B, T, D)
        self.inject_at: Dict[int, Tuple[List[int], torch.Tensor]] = {}
        # ^ inject_at[layer_idx] = (positions, tensor of shape (len(positions), D))

    # ---- recording (original pass) ----
    def _make_record_hook(self, idx: int):
        def hook(_module, inputs, _kwargs_or_output=None):
            # pre-forward hook: inputs[0] is the hidden states entering this layer
            hs = inputs[0]
            self.recorded[idx] = hs.detach()
        return hook

    def start_recording(self):
        self.clear_handles()
        self.recorded = {}
        for i, layer in enumerate(self.layers):
            self.handles.append(layer.register_forward_pre_hook(self._make_record_hook(i)))

    # ---- injecting (interpretation pass) ----
    def _make_inject_hook(self, idx: int):
        def hook(_module, inputs):
            if idx not in self.inject_at:
                return None
            positions, vec = self.inject_at[idx]
            hs = inputs[0]
            # Only inject on the first (prefill) step, when the sequence dim is
            # the full prompt length. During generation we see seq_len==1 (cached
            # decoding) and we must NOT rewrite that; we've already rewritten
            # the layer-k KV cache implicitly by rewriting the prefill hidden.
            if hs.shape[1] <= max(positions):
                return None
            # hs: (B, T, D); write the same vec to every item in the batch
            hs = hs.clone()  # never mutate caller's tensor
            for p in positions:
                hs[:, p, :] = vec.to(hs.dtype).to(hs.device)
            # reconstruct the input tuple with the edited tensor
            new_inputs = (hs,) + inputs[1:]
            return new_inputs
        return hook

    def start_injecting(self, inject_at: Dict[int, Tuple[List[int], torch.Tensor]]):
        self.clear_handles()
        self.inject_at = inject_at
        for i, layer in enumerate(self.layers):
            self.handles.append(layer.register_forward_pre_hook(self._make_inject_hook(i)))

    # ---- cleanup ----
    def clear_handles(self):
        for h in self.handles:
            h.remove()
        self.handles = []


# ---------------------------------------------------------------------------
# 3. Public API
# ---------------------------------------------------------------------------

@torch.no_grad()
def record_hidden_states(
    model: nn.Module,
    *,
    input_ids: Optional[torch.LongTensor] = None,
    attention_mask: Optional[torch.Tensor] = None,
    pixel_values: Optional[torch.Tensor] = None,
    **extra_forward_kwargs,
) -> List[torch.Tensor]:
    """Run one forward pass and return a list of hidden states, one per layer.

    The returned list has length num_layers; entry i is the hidden state ENTERING
    layer i, shape (B, T, D). This matches SelfIE's `outputs['hidden_states'][l]`
    for l in [0, num_layers-1]. (We ignore the final post-norm output since SelfIE
    injects at layers, not at the lm_head.)
    """
    layers = _get_llama_layers(model)
    injector = _Injector(layers)
    injector.start_recording()
    try:
        call_kwargs = dict(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=False,   # we use our own hooks
            use_cache=False,              # keep things simple for a single forward
            return_dict=True,
        )
        if pixel_values is not None:
            call_kwargs["pixel_values"] = pixel_values
        call_kwargs.update(extra_forward_kwargs)
        model(**call_kwargs)
    finally:
        injector.clear_handles()

    # Return layer inputs in order.
    return [injector.recorded[i] for i in range(len(layers))]


@torch.no_grad()
def interpret_embedding(
    model: nn.Module,
    tokenizer,
    embedding: torch.Tensor,            # shape (D,) or (1, 1, D)
    interp_prompt: InterpretationPrompt,
    *,
    inject_layer: int = 2,
    max_new_tokens: int = 30,
    do_sample: bool = False,
) -> str:
    """Interpret a single hidden-state vector by running the interpretation
    forward pass with hooks that overwrite the placeholder hidden states at
    `inject_layer` with `embedding`.

    Returns the generated text, stripped of the prompt.
    """
    layers = _get_llama_layers(model)
    D = embedding.numel()
    vec = embedding.reshape(D).to(next(model.parameters()).device)

    inject_at = {inject_layer: (interp_prompt.placeholder_positions, vec)}

    injector = _Injector(layers)
    injector.start_injecting(inject_at)
    try:
        prompt_len = interp_prompt.input_ids.shape[1]
        gen = model.generate(
            input_ids=interp_prompt.input_ids.to(model.device),
            attention_mask=interp_prompt.attention_mask.to(model.device),
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id,
        )
    finally:
        injector.clear_handles()

    new_tokens = gen[0, prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


@torch.no_grad()
def interpret_openvla(
    openvla_model,
    processor,
    image,                               # PIL Image or None for text-only
    prompt: str,                         # e.g. "In: What action should the robot take to pick up the red block?\nOut:"
    *,
    tokens_to_interpret: Sequence[Tuple[int, int]],  # list of (retrieve_layer, retrieve_token)
    interp_prompt: Optional[InterpretationPrompt] = None,
    inject_layer: int = 2,
    max_new_tokens: int = 30,
) -> List[dict]:
    """End-to-end: record hidden states from an OpenVLA forward pass on
    (image, prompt), then for each (layer, token) pair, ask OpenVLA's own LLM
    to describe that embedding in words.

    Returns a list of dicts with keys: layer, token, token_decoded, interpretation.
    """
    tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor

    # --- interpretation prompt default (Llama-2-chat style; OpenVLA uses Llama-2) ---
    if interp_prompt is None:
        interp_prompt = InterpretationPrompt.build(
            tokenizer,
            ("[INST] ", 0, 0, 0, 0, 0, " [/INST] Sure, I will summarize the message:"),
        )

    # --- 1. original forward on the real (image, text) input ---
    if image is not None:
        inputs = processor(prompt, image)
    else:
        inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(openvla_model.device) for k, v in inputs.items()
              if isinstance(v, torch.Tensor)}

    all_hidden = record_hidden_states(openvla_model, **inputs)

    # Decode every token in the input for logging (OpenVLA prepends image patch
    # tokens AFTER the BOS token, so there's no text equivalent for those).
    input_ids = inputs.get("input_ids")
    decoded = {}
    if input_ids is not None:
        for t in range(input_ids.shape[1]):
            decoded[t] = tokenizer.decode(input_ids[0, t])

    # --- 2. interpretation pass per (layer, token) ---
    results = []
    for l, t in tokens_to_interpret:
        if l >= len(all_hidden):
            raise IndexError(f"retrieve_layer={l} >= num_layers={len(all_hidden)}")
        if t >= all_hidden[l].shape[1]:
            raise IndexError(
                f"retrieve_token={t} >= seq_len={all_hidden[l].shape[1]}. "
                f"Remember OpenVLA prepends {all_hidden[l].shape[1] - input_ids.shape[1]} "
                f"image-patch tokens after <BOS>."
            )
        vec = all_hidden[l][0, t]  # (D,)
        text = interpret_embedding(
            openvla_model, tokenizer, vec, interp_prompt,
            inject_layer=inject_layer, max_new_tokens=max_new_tokens,
        )
        results.append({
            "layer": l,
            "token": t,
            "token_decoded": decoded.get(t, "<image_patch_token>"),
            "interpretation": text.strip(),
        })

    return results


# ---------------------------------------------------------------------------
# 4. How to patch SelfIE itself if you insist on using its upstream code
# ---------------------------------------------------------------------------

def patch_selfie_for_transformers_4_40():
    """Documentation-only helper. If you want to use the upstream SelfIE repo
    (https://github.com/tonychenxyz/selfie) against transformers 4.40.1 (which
    is what OpenVLA pins), you need the following edits to SelfIE's
    `selfie/llama_forward_wrappers.py`. These are the minimum edits; after them
    the module will import and run on transformers 4.38 - 4.44.

    ===== Edit 1: flash-attn symbol rename =====
        Old (line ~37):
            from transformers.utils import ( ..., is_flash_attn_available, ... )
        New:
            from transformers.utils import ( ..., is_flash_attn_2_available as is_flash_attn_available, ... )
        (or just delete the import + its guarded block; SelfIE never actually
         calls flash-attn in its interpretation passes.)

    ===== Edit 2: attention-mask preparation =====
        Old (line ~313):
            attention_mask = model.model._prepare_decoder_attention_mask(
                attention_mask, (batch_size, seq_length), inputs_embeds, past_key_values_length
            )
        New:
            from transformers.modeling_attn_mask_utils import _prepare_4d_causal_attention_mask
            attention_mask = _prepare_4d_causal_attention_mask(
                attention_mask, (batch_size, seq_length), inputs_embeds, past_key_values_length
            )

    ===== Edit 3: drop the removed `padding_mask=` kwarg =====
        Old (line ~441, inside decoder_layer_forward_interpret):
            hidden_states, self_attn_weights, present_key_value = decoder_layer.self_attn(
                hidden_states=hidden_states,
                attention_mask=attention_mask,
                position_ids=position_ids,
                past_key_value=past_key_value,
                output_attentions=output_attentions,
                use_cache=use_cache,
                padding_mask=padding_mask,        # <-- remove this line
            )
        New: just delete the `padding_mask=padding_mask,` line. The `padding_mask`
        kwarg was removed from LlamaAttention.forward in transformers 4.38.

        Also remove the corresponding `padding_mask=padding_mask` in the
        gradient-checkpointing branch around line 356.

    ===== Edit 4 (only if you want use_cache=True): DynamicCache migration =====
        From transformers 4.36 onward, `past_key_values` is a `DynamicCache`
        object, not a tuple of tuples. SelfIE's line
            past_key_values_length = past_key_values[0][0].shape[2]
        blows up. Safest fix: force `use_cache=False` in SelfIE's interpretation
        forward, since it runs a full forward on the interpretation prompt each
        time anyway. Concretely, at the top of `model_model_forward_interpret`:
            use_cache = False          # <-- add this line; SelfIE doesn't need
                                       #     KV cache during interpretation.

    With those four edits, SelfIE runs correctly against OpenVLA's pinned
    transformers 4.40.1. However, the hook-based approach in this file is
    recommended because:
      * It is version-agnostic (no forward-method copy).
      * It does not mutate SelfIE source, so you can keep SelfIE pinned and
        just install its utilities.
      * OpenVLA uses flash-attn-2 during normal inference; SelfIE's hand-rolled
        forward bypasses the flash path, which silently changes numerics. Hooks
        keep the original flash path intact.
    """
    # This function is pure documentation; nothing to execute.
    return None


## Cell 6 — Load OpenVLA-7B

First run: 5-10 minute download. With Drive caching: ~30s on subsequent runs.

On A100: uses ~14 GB VRAM. On V100 (16 GB): tight but fits — close other notebooks.

In [ ]:
import torch
from transformers import AutoModelForVision2Seq, AutoProcessor

processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    # attn_implementation="flash_attention_2",  # uncomment if you installed flash-attn in cell 4
).to("cuda:0").eval()

print(f"Loaded. GPU mem used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(f"Llama backbone layers: {len(model.language_model.model.layers)}")
print(f"Hidden size: {model.language_model.config.hidden_size}")

/home/meek/.pyenv/versions/3.10.13/envs/openvla-selfie/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/meek/.pyenv/versions/3.10.13/envs/openvla-selfie/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/openvla/openvla-7b:
- processing_prismatic.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/home/meek/.pyenv/versions/3.10.13/envs/openvla-selfie/lib/python3.10/site-packages/huggingface_hub/file_dow

## Cell 7 — Pick a test image and instruction

Three options. Run exactly one of them.

### 7A — Use a Bridge validation example (recommended, in-distribution for openvla-7b)

In [ ]:
from datasets import load_dataset

# Small public Bridge-style (WidowX) scene set — ~660 KB, 12 images, imagefolder format.
# Returns PIL images directly. Swap in any other PIL-image-serving HF dataset if you prefer.
ds = load_dataset("Sombit/v3la_eval_database_widowx", split="train", streaming=True)
example = next(iter(ds))

image = example["image"].convert("RGB")    # PIL image (320x240)

# The scene label encodes the task; map to a natural-language instruction.
# Override `instruction` below with any known-good Bridge task if you want.
label_names = ["corn_between_cups", "stack_pink_blue_cup"]
scene_instructions = {
    "corn_between_cups":   "put the yellow corn between the cups",
    "stack_pink_blue_cup": "stack the pink cup on the blue cup",
}
scene = label_names[example["label"]]
instruction = scene_instructions[scene]    # e.g. "put the yellow corn between the cups"

print("Scene:", scene)
print("Using instruction:", instruction)
image  # displays inline

### 7B — Use your own uploaded image

If you uploaded an image via the Files panel (left sidebar), use this cell instead of 7A.

In [ ]:
# from PIL import Image
# image = Image.open("/content/my_scene.jpg").convert("RGB")
# instruction = "pick up the red block and place it on the plate"
# image

### 7C — Load an image from a URL

Any known-good Bridge instruction works with a Widow-X-style tabletop scene:
- "put the eggplant in the pot"
- "put the yellow corn on the pink plate"
- "lift the red chili pepper"
- "pick up the blue cup"

## Cell 8 — Run the interpretation

This is the actual SelfIE pipeline on OpenVLA.

We interpret the **center image patch** across several layers. OpenVLA's image-patch tokens occupy positions 1..256 in a 16×16 grid. Position 137 ≈ center patch (row 8, col 8).

In [ ]:
from openvla_selfie import interpret_openvla, InterpretationPrompt

prompt = f"In: What action should the robot take to {instruction}?\nOut:"

# OpenVLA's image-patch grid: 16x16 = 256 patches, positions 1..256 in the Llama stream
SQRT = 16
BASE = 1
def patch_pos(row, col):
    return BASE + row * SQRT + col

center_patch = patch_pos(SQRT // 2, SQRT // 2)     # center of image
corner_patches = [patch_pos(0, 0), patch_pos(0, SQRT-1),
                  patch_pos(SQRT-1, 0), patch_pos(SQRT-1, SQRT-1)]

# Trace the center patch across several layers
tokens_to_interpret = [(l, center_patch) for l in (5, 10, 15, 20, 25)]

# Build an interpretation prompt. You can also let interpret_openvla use the default.
interp = InterpretationPrompt.build(
    processor.tokenizer,
    ("[INST] ", 0, 0, 0, 0, 0, " [/INST] This hidden state represents the concept:"),
)

# Important: ban action tokens during interpretation so OpenVLA doesn't emit CJK gibberish
# from the overwritten last-256 vocab entries. These IDs are the 256 action tokens.
BAD_WORDS = [[i] for i in range(processor.tokenizer.vocab_size - 256, processor.tokenizer.vocab_size)]

results = interpret_openvla(
    model, processor, image, prompt,
    tokens_to_interpret=tokens_to_interpret,
    interp_prompt=interp,
    inject_layer=2,
    max_new_tokens=25,
)

print(f"Instruction: {instruction}")
print(f"Interpreting center image patch (pos {center_patch}) across layers\n")
print("=" * 78)
for r in results:
    print(f"layer {r['layer']:>2d}  tok {r['token']:>4d}  -> {r['interpretation']}")

## Cell 9 — Spatial scan: interpret 5 patches at the same layer

This shows *which image regions encode what* at a mid-depth layer. Run after cell 8 works.

In [ ]:
# Five spatial samples: center + four corners
scan_layer = 20
tokens_spatial = [(scan_layer, p) for p in [center_patch] + corner_patches]
labels = ["center", "top-left", "top-right", "bottom-left", "bottom-right"]

results = interpret_openvla(
    model, processor, image, prompt,
    tokens_to_interpret=tokens_spatial,
    interp_prompt=interp,
    inject_layer=2,
    max_new_tokens=25,
)

print(f"Spatial scan at layer {scan_layer}:\n")
for label, r in zip(labels, results):
    print(f"  {label:<15} (pos {r['token']:>4d})  ->  {r['interpretation']}")

## Cell 10 — Interpret the "action intent" token

The last text token (right before OpenVLA starts emitting action tokens) holds the hidden state that directly produces the first action dimension (dx). Interpreting it is the closest thing to asking OpenVLA "what are you about to do, in English?".

In [ ]:
# Figure out the actual sequence length after the processor inserts image patches
inputs = processor(prompt, image)
seq_len = inputs["input_ids"].shape[1]
# With 256 image patches inserted after BOS, total = 1 + 256 + len(text tokens)
last_text_pos = seq_len - 1
print(f"Sequence length: {seq_len}")
print(f"Last text position (the 'intent' token): {last_text_pos}")

intent_results = interpret_openvla(
    model, processor, image, prompt,
    tokens_to_interpret=[(l, last_text_pos) for l in (15, 20, 25, 29)],
    interp_prompt=interp,
    inject_layer=2,
    max_new_tokens=30,
)

print(f"\nInterpreting the 'action intent' hidden state:")
print("=" * 78)
for r in intent_results:
    print(f"  layer {r['layer']:>2d}  ->  {r['interpretation']}")

## What to look for

- **Cell 8**: mid-to-deep layers should describe what's in the center of the image. Early layers (5, 10) may be vague; by layer 20-25 you should see object-relevant words.
- **Cell 9**: different spatial patches at the same layer should produce different descriptions — corners often hit "table surface" or "background," center hits the target object.
- **Cell 10**: the action-intent token should produce something motion-relevant like "reach down and grasp" or "move to the left."

If interpretations look like CJK characters or random unicode, it's action-token leakage — OpenVLA's generate() sampled from the 256 action-token vocab slots. Fix by passing `bad_words_ids=BAD_WORDS` to generate; I left BAD_WORDS defined in cell 8 for exactly this.

If interpretations are coherent English but uninformative — try lowering `inject_layer` to 1, or raising `max_new_tokens` to 50.

If interpretations are all identical regardless of which embedding you feed in — check GPU memory with `nvidia-smi`; you may be silently falling back to CPU.